# Variable-Channel Spatial ASR — 학습·검증 노트북

화자 구분 비교 사다리의 모든 시스템을 학습하고 검증 세트에서 임계값과 최고 체크포인트를 정합니다. 본 평가셋(`eval-t2`, `eval-t2-static`) 채점은 [`benchmark.ipynb`](https://colab.research.google.com/github/hd1534/variable-channel-spatial-asr/blob/main/notebooks/benchmark.ipynb)가 맡습니다. 벤치마크 노트북은 이 노트북이 개인 Drive에 남긴 `runs/<시스템>`을 읽으므로, 이 노트북이 돌고 있는 동안 다른 런타임에서 끝난 시스템부터 채점할 수 있습니다.

- **`CONFIG = "all"` (기본값)**: 비교군 3종(`voice_only` → `space_only` → `naive_fusion`)과 제안 모델 3종(`proposed_carry` → `proposed_reset` → `proposed_carry_nodrop`)을 **한 번의 실행으로 순서대로 전부** 학습·검증합니다.
- **중간 종료 후 완료된 백업에서 재개합니다**: 체크포인트, 완료된 `summary.json`, RIR 뱅크, 데이터 캐시를 개인 Drive로 백업합니다. 복원 기준은 마지막 완료된 백업입니다. 런타임이 초기화되면 **그냥 다시 `런타임 > 모두 실행`**만 누르세요. 이미 끝난 모델은 건너뛰고, 학습 중이던 모델은 마지막 체크포인트에서 자동으로 이어서 진행합니다. 단, 그 사이 저장소에 새 커밋을 올렸다면 끝나지 않은 제안 모델 학습은 `ALLOW_CODE_CHANGE = True` 없이는 이어 가지 않고 멈춥니다(4절). 목표 걸음 수까지 끝난 실행은 막히지 않으며, 검증 기준이 바뀌었으면 더 학습하지 않고 새 코드로 다시 검증합니다.

| CONFIG | 시스템 | 학습 여부 |
|---|---|---|
| `all` | **전체 6개 본 실험 시스템 순차 자동 실행** (`voice_only` ~ `proposed_carry_nodrop`) | 자동 |
| `all_baselines` | 무학습 비교군 3종(`voice_only`, `space_only`, `naive_fusion`)만 순차 실행 | 임계값·가중치만 조정 |
| `all_proposed` | 제안 모델 3종(`proposed_carry`, `proposed_reset`, `proposed_carry_nodrop`)만 순차 실행 | 학습 |
| `smoke_real` | GPU 사전 점검: 실제 ECAPA · pyroomacoustics · 실제 크기 모델로 30걸음 (약 5~10분) | 학습 (점검용) |
| `voice_only` | 목소리 전용: 동결 ECAPA + 온라인 군집 | 임계값만 조정 |
| `space_only` | 공간 전용: SRP-PHAT 방위 분포 + 온라인 군집 | 임계값만 조정 |
| `naive_fusion` | 단순 결합: 두 유사도를 고정 가중치로 합산 | 가중치·임계값만 조정 |
| `proposed_carry` | 제안 모델: TAC + ECAPA + GRU, 청크 간 상태 유지 (Carry-Over) | 학습 |
| `proposed_reset` | 제안 모델: 같은 구조, 청크마다 상태 초기화 (Chunk-Reset) | 학습 |
| `proposed_carry_nodrop` | 제안 모델 대조군: 4채널 고정 학습 (채널 드롭아웃 없음) | 학습 |

저장소: https://github.com/hd1534/variable-channel-spatial-asr

## 0. 설정

그대로 두고 `런타임 > 모두 실행`을 누르면 전체 실험(`all`)과 본 평가셋 채점(`RUN_EVAL_T2 = True`)이 한 번에 돌아갑니다.

In [ ]:
STORAGE_BACKEND: str = "drive_api"  # @param ["drive_api", "mounted_drive"]
DRIVE_SECRET: str = "DRIVE_AUTH"  # @param {type:"string"}
LOCAL_STORE_ROOT: str = "/content/vcsasr-store"
BACKUP_EVERY_SEC: int = 120  # @param {type:"integer"}
# 이 VM으로 내려받지 않을 백업 경로 접두어 목록. None이면 기본값(mono 비교군 전용 가중치)을 쓰고 []이면 모두 받습니다.
# 건너뛴 파일도 백업에는 그대로 남습니다.
RESTORE_SKIP: list[str] | None = None
# 학교 Drive의 기존 결과를 한 번 가져올 경로. 예: /content/drive/MyDrive/vcsasr (백업에 없는 폴더만 복사합니다)
IMPORT_FROM: str = ""  # @param {type:"string"}

CONFIG = "all"  # @param ["all", "all_baselines", "all_proposed", "smoke_real", "smoke", "voice_only", "space_only", "naive_fusion", "proposed_carry", "proposed_reset", "proposed_carry_nodrop"]
# 학습 화자. train-clean-100(251명, 6.3 GB)이 기본입니다. dev-clean-only는 큰 내려받기 없이
# dev-clean의 평가 제외 화자 34명을 학습 24명 · 검증 10명으로 나눠 빨리 돌려 보는 용도입니다.
SPEAKERS = "train-clean-100"  # @param ["train-clean-100", "dev-clean-only"]
# 설정 덮어쓰기 (공백으로 구분). 예: train.max_steps=2000 train.batch_size=8
# 제안 모델에서 train.max_steps를 바꾸면 실행 이름에 -steps<N>이 붙어 본 학습과 다른 폴더(runs/<CONFIG>-steps<N>)에 저장됩니다.
# 실행 이름에 꼬리표(-devclean, -<RUN_TAG>, -steps<N>)가 붙으면 benchmark.ipynb의 RUN_SUFFIX에 같은 꼬리표를 넣습니다.
OVERRIDES = ""  # @param {type:"string"}
RUN_TAG = ""  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
COMMIT = ""  # @param {type:"string"}
ALLOW_CODE_CHANGE = False  # @param {type:"boolean"}

REPO_URL = "https://github.com/hd1534/variable-channel-spatial-asr.git"
DRIVE_ROOT: str = LOCAL_STORE_ROOT if STORAGE_BACKEND == "drive_api" else "/content/drive/MyDrive/vcsasr"
DATA_ROOT = "/content/data"                   # 학습 중 읽는 데이터 (로컬 디스크가 Drive보다 훨씬 빠르다)
REPO_DIR = "/content/variable-channel-spatial-asr"

PRESETS = {
    "all": ["voice_only", "space_only", "naive_fusion", "proposed_carry", "proposed_reset", "proposed_carry_nodrop"],
    "all_baselines": ["voice_only", "space_only", "naive_fusion"],
    "all_proposed": ["proposed_carry", "proposed_reset", "proposed_carry_nodrop"],
}
CONFIG_LIST = PRESETS.get(CONFIG, [CONFIG])
print("실행할 CONFIG 목록:", CONFIG_LIST)

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
from typing import Any

if "storage" not in globals():
    storage: Any | None = None


def run_local(cmd: str, check: bool = True) -> int:
    """백업 세션 없이 셸 명령을 실행한다. 백업을 연결하기 전의 git·pip 명령과 mounted_drive 모드에만 쓴다."""
    print("$", cmd, flush=True)
    env = dict(os.environ, PYTHONUNBUFFERED="1")
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                            env=env)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0 and check:
        raise RuntimeError(f"실패 (종료 코드 {proc.returncode}): {cmd}")
    return proc.returncode


def require_backup() -> None:
    """drive_api 모드에서 백업 세션 없이(예: 재연결 실패 후) 작업을 이어 가지 않도록 멈춘다."""
    if STORAGE_BACKEND == "drive_api" and storage is None:
        raise RuntimeError("개인 Drive 백업 세션이 없습니다. 연결 셀의 오류를 해결한 뒤 그 셀부터 다시 실행하세요.")


def sh(cmd: str, check: bool = True) -> int:
    """셸 명령을 실행하고 출력을 셀에 바로 보여 준다. 실패하면 셀을 멈춘다."""
    require_backup()
    if storage is None:
        return run_local(cmd, check)
    print("$", cmd, flush=True)
    return storage.run(cmd, shell=True, check=check)


def drop_repo_modules(repo_dir: str) -> int:
    """저장소 파일에서 불러온 모듈을 sys.modules에서 지워, 이후 import가 새로 체크아웃한 코드를 쓰게 한다."""
    root = Path(repo_dir).resolve()
    stale: list[str] = []
    for name, module in list(sys.modules.items()):
        try:
            paths = [getattr(module, "__file__", None), *(getattr(module, "__path__", None) or [])]
            inside = any(isinstance(p, str) and os.path.isabs(p) and Path(p).resolve().is_relative_to(root)
                         for p in paths)
        except Exception:  # 속성을 읽을 수 없는 특수 모듈은 저장소 모듈이 아니다.
            continue
        if inside and name != "__main__":
            stale.append(name)
    for name in stale:
        del sys.modules[name]
    return len(stale)


IN_COLAB = "google.colab" in sys.modules
import torch

GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print(f"Colab: {IN_COLAB} | torch {torch.__version__} | GPU: {GPU or '없음'}"
      + (f" | compute capability {torch.cuda.get_device_capability()}" if GPU else ""))
if IN_COLAB and GPU is None and any(c.startswith("proposed") or c == "smoke_real" for c in CONFIG_LIST):
    raise RuntimeError("GPU가 없습니다. 런타임 > 런타임 유형 변경에서 GPU를 고른 뒤 다시 실행하세요.")

## 개인 Drive 연결 (처음 한 번)

학교 계정의 Colab Pro+를 유지하고 개인 계정의 Drive를 **Google 공식 Python 라이브러리**로 사용합니다.
[개인 Drive 설정 가이드](https://github.com/hd1534/variable-channel-spatial-asr/blob/main/docs/personal-drive.md)에 따라
Google Cloud 프로젝트에서 Drive API와 Desktop OAuth 클라이언트를 만들고, Mac에서 `python3 setup_drive.py`를 실행하세요.
생성된 `runs/private/drive-auth.json` 내용을 **Colab 왼쪽 열쇠 아이콘 → 보안 비밀 → `DRIVE_AUTH`**에 저장하고 노트북 접근을 허용합니다.
인증 파일·토큰은 채팅, 코드 셀, 출력, Git에 붙여 넣지 않습니다. 기존 rclone 설정은 사용하지 않습니다.

`STORAGE_BACKEND="drive_api"`가 기본입니다. 학교 Drive를 마운트하지 않으며, 파일은 VM의 `LOCAL_STORE_ROOT`에서 읽고 씁니다.
재접속하면 마지막 완료 백업을 검증해 복원합니다. 같은 계정의 일반 Drive 마운트는 `mounted_drive`로 선택할 수 있습니다.
OAuth 앱이 **Testing** 상태이면 갱신 토큰이 7일 뒤 만료되므로, 장기 사용 전 가이드의 게시 상태 안내도 확인하세요.

여러 런타임이 같은 백업 폴더를 동시에 써도 됩니다. 런타임마다 자기 백업 이력을 이어 가고, 연결 셀과 명령이 시작할 때마다 다른 런타임의 이력과 합칩니다.
한 런타임만 바꾼 폴더는 그 런타임의 버전을 씁니다. 같은 `runs/<실행>` 폴더를 두 런타임이 모두 바꿨다면 파일을 더 나중에 쓴 버전을 쓰고, 다른 버전은 백업 이력에 남깁니다(가이드의 `export --commit`으로 꺼냅니다). 그래서 이 노트북과 `benchmark.ipynb`를 서로 다른 런타임에서 동시에 돌려도 됩니다.
다만 같은 설정을 두 런타임에서 동시에 학습하지는 마세요. GPU 시간을 두 번 쓰고, 한쪽 결과는 최신 버전에서 빠집니다.
**이전 코드에서 올릴 때**: 이전 커밋(`9123e53`)의 코드로 돌고 있는 런타임은 다른 런타임이 백업을 올리면 다음 주기 백업에서 `remote advanced in another runtime` 오류로 작업을 멈춥니다. 새 코드의 연결 셀은 연결하자마자 백업을 올리므로, 새 코드로 다른 런타임을 연결하기 전에 이전 코드로 돌고 있는 런타임을 모두 멈추거나 새 코드로 다시 실행하세요.

`RESTORE_SKIP`은 이 VM으로 내려받지 않을 경로 접두어 목록입니다. 기본값(`None`)은 학습에 쓰지 않는 본 평가 결과(`eval`, `mono_baselines`), 전사 캐시, 전사기·단일채널 비교군 가중치(Qwen3-ASR, wav2vec2, Nemotron, VibeVoice, MOSS, VibeVoice 토크나이저 `Qwen2.5-7B`)를 건너뜁니다. `[]`이면 모두 받습니다. 건너뛴 파일도 백업에는 그대로 남습니다.

학교 Drive에 있던 기존 결과는 `IMPORT_FROM`(예: `/content/drive/MyDrive/vcsasr`)으로 한 번 가져옵니다. 연결 셀이 학교 Drive를 마운트하고, 개인 백업에 아직 없는 폴더(`runs/<실행>`, `eval/<이름>`, `rirbanks/<뱅크>`, `cache/` 파일 등)만 복사합니다. 이미 있는 폴더는 덮어쓰지 않으며(ASR 전사 캐시는 실행 계약과 세션 오디오 해시가 같을 때만 합치고, 다르면 백업의 캐시를 그대로 둡니다), 가져온 뒤에는 비워 둡니다. 가져오지 않으면 빈 백업에서 시작해 모든 시스템을 처음부터 학습합니다.

120초마다 학습 결과를 백업하고, 명령 종료 시 대용량 모델·코퍼스 캐시도 백업합니다.
업로드한 크기·체크섬을 확인한 뒤에만 완료 기록을 저장합니다. 전송 실패는 셀 오류로 표시합니다. 연결 셀이 실패하면 이후 명령 셀도 백업 없이 실행하지 않고 오류로 멈춥니다.
갑작스러운 런타임 종료 시 **마지막 완료 백업 이후 작업은 유실될 수 있습니다**. 마지막 셀의 `개인 Drive 최종 백업 완료`까지 확인하세요.


In [ ]:
if storage is not None:
    try:
        storage.close()
    except RuntimeError as exc:
        print(f"이전 백업 세션 종료: {exc}", flush=True)
    storage = None

# 백업 세션은 아래에서 연결하므로 그 전의 git·pip 명령은 run_local로 실행한다.
if not Path(REPO_DIR, ".git").is_dir():
    run_local(f'git clone -q --depth 1 --filter=blob:none --no-checkout -b {BRANCH} {REPO_URL} "{REPO_DIR}"')
    # 평가셋 오디오는 학습에 쓰지 않는다 (본 평가는 benchmark.ipynb).
    run_local(f"git -C \"{REPO_DIR}\" sparse-checkout set --no-cone '/*' '!/data/*/audio/'")
run_local(f'git -C "{REPO_DIR}" fetch -q --depth 1 origin {COMMIT or BRANCH} && git -C "{REPO_DIR}" checkout -q FETCH_HEAD')
os.chdir(REPO_DIR)
run_local("git log -1 --format='%h %s'")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
# 같은 커널에서 새 커밋을 받았다면 이전 코드로 불러온 프로젝트 모듈을 비워, 아래 import부터 새 코드를 쓴다.
print(f"이전 코드로 불러온 프로젝트 모듈 {drop_repo_modules(REPO_DIR)}개를 비웠습니다.", flush=True)

if STORAGE_BACKEND == "drive_api":
    from drive_storage import connect_personal_drive, model_cache_dirs
    run_local(f'{sys.executable} -m pip -q install -r requirements-drive.txt')
    if RESTORE_SKIP is None:
        # 본 평가(benchmark.ipynb)의 결과·전사 캐시·전사기 가중치는 학습에 쓰지 않는다 (Qwen2.5-7B는 VibeVoice 토크나이저).
        RESTORE_SKIP = ["eval", "mono_baselines", "asr_cache", "asr_comparison.json"] + model_cache_dirs(
            ["Qwen/Qwen3-ASR-1.7B", "facebook/wav2vec2-base-960h", "nvidia/Nemotron-3-Diarization",
             "microsoft/VibeVoice-ASR", "OpenMOSS-Team/MOSS-Transcribe-Diarize", "Qwen/Qwen2.5-7B"])
    print("복원 제외 (백업에는 남음):", RESTORE_SKIP, flush=True)
    import_from: Path | None = None
    if IMPORT_FROM:
        if "google.colab" in sys.modules:
            from google.colab import drive
            drive.mount("/content/drive")
        import_from = Path(IMPORT_FROM)
        if not import_from.is_dir():
            raise FileNotFoundError(f"IMPORT_FROM 폴더가 없습니다: {IMPORT_FROM}")
        print(f"백업에 없는 결과 폴더를 {import_from}에서 가져옵니다.", flush=True)
    storage = connect_personal_drive(Path(LOCAL_STORE_ROOT), secret_name=DRIVE_SECRET, interval_sec=BACKUP_EVERY_SEC,
                                     skip=RESTORE_SKIP, import_from=import_from)
elif STORAGE_BACKEND == "mounted_drive":
    if IMPORT_FROM:
        raise ValueError('IMPORT_FROM은 STORAGE_BACKEND="drive_api"에서만 씁니다. mounted_drive는 마운트한 Drive를 바로 씁니다.')
    if "google.colab" in sys.modules:
        from google.colab import drive
        drive.mount("/content/drive")
    Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
else:
    raise ValueError("STORAGE_BACKEND를 확인하세요.")
print(f"저장소: {STORAGE_BACKEND} | 작업 경로: {DRIVE_ROOT}", flush=True)


## 2. 패키지 설치

Colab에 미리 설치된 `torch`·`torchaudio`·`numpy`는 그대로 두고(제약 파일로 고정), 동결 화자 인코더(ECAPA-TDNN)를
불러올 `speechbrain`만 설치합니다. 방 음향을 계산하는 `pyroomacoustics`는 Drive에 방 음향 뱅크가 없을 때만 3절에서 설치합니다.

In [ ]:
sh("pip freeze | grep -iE '^(torch|torchaudio|numpy)==' > /content/constraints.txt")
sh('pip -q install -c /content/constraints.txt "speechbrain==1.1.2"')
sh(f'{sys.executable} -c "import torchaudio, soundfile; from speechbrain.inference.speaker import EncoderClassifier; '
   f'import torch; print(\'torch\', torch.__version__, \'| speechbrain 설치 확인\')"')

# Cache public Hugging Face weights without moving authentication tokens into the backup tree.
os.environ["HF_HUB_CACHE"] = str(Path(DRIVE_ROOT, "models", "huggingface"))


## 3. 데이터 및 방 음향 뱅크 준비

- **LibriSpeech**: 로컬 `cache/`의 압축 파일을 `/content/data`에 풉니다. 명령 종료 시 개인 Drive에도 백업합니다.
- **방 음향 뱅크**: 학습용 800개·검증용 64개 방을 생성합니다. 로컬 `rirbanks/`에 완성된 조각을 보관하고 주기적으로 백업합니다.
- **ECAPA 및 ASR 체크포인트**: 로컬 `models/`에서 사용하고 명령 종료 시 백업합니다.


In [ ]:
LIBRISPEECH_MD5 = {
    "dev-clean": "42e2234ba48799c1f50f24a7926300a1",
    "train-clean-100": "2a93770f6d5c6c964bc36631d331a522",
}
MIRRORS = ["https://www.openslr.org/resources/12", "https://openslr.elda.org/resources/12",
           "https://openslr.trmal.net/resources/12"]

from train import _parse_set
from training.config import RunConfig, load_config

def build_run_info(cfg_name: str) -> tuple[str, str, str, bool, RunConfig]:
    """Build one final run configuration and its persistent working paths."""
    cfg_json = json.loads(Path(f"configs/{cfg_name}.json").read_text(encoding="utf-8"))
    override_tokens = OVERRIDES.split()
    override_dict = _parse_set(override_tokens)
    synthetic = bool(override_dict.get("data.synthetic", cfg_json.get("data", {}).get("synthetic")))
    own_train_root = "data.train_root" in override_dict or "train_root" in cfg_json.get("data", {}) or synthetic
    dev_only = SPEAKERS == "dev-clean-only" and not own_train_root
    if dev_only:
        override_tokens.append("data.train_root=LibriSpeech/dev-clean")
        override_dict = _parse_set(override_tokens)
    cfg = load_config(f"configs/{cfg_name}.json", override_dict)
    sets = [f"--set {o}" for o in override_tokens]
    run_name = cfg_name + ("-devclean" if dev_only else "") + (f"-{RUN_TAG}" if RUN_TAG else "")
    if cfg.model.name == "proposed" and "train.max_steps" in override_dict:
        # 짧은 시간 측정 실행이 본 학습의 runs/<CONFIG>를 이어 쓰지 않도록 걸음 수를 이름에 넣는다.
        run_name += f"-steps{cfg.train.max_steps}"
    run_dir = f"{DRIVE_ROOT}/runs/{run_name}"
    args = (f'--config configs/{cfg_name}.json --data-root "{DATA_ROOT}" --run-dir "{run_dir}" '
            f'--model-cache "{DRIVE_ROOT}/models/ecapa" --bank-mirror "{DRIVE_ROOT}/rirbanks" '
            + ("--allow-code-change " if ALLOW_CODE_CHANGE else "") + " ".join(sets))
    return run_name, run_dir, args, dev_only, cfg

need = []
for c in CONFIG_LIST:
    _, _, _, dev_only, cfg = build_run_info(c)
    trainable = cfg.model.name == "proposed"
    if not cfg.data.synthetic and "dev-clean" not in need:
        need.append("dev-clean")
    if trainable and not cfg.data.synthetic and cfg.data.train_root == "LibriSpeech/train-clean-100" and "train-clean-100" not in need:
        need.append("train-clean-100")

cache = Path(DRIVE_ROOT, "cache")
cache.mkdir(parents=True, exist_ok=True)
Path(DATA_ROOT).mkdir(parents=True, exist_ok=True)
for name in need:
    if Path(DATA_ROOT, "LibriSpeech", name).is_dir():
        print("이미 풀려 있음:", name)
        continue
    cached = cache / f"{name}.tar.gz"
    if cached.is_file():
        tar = cached
    else:
        tar = Path(DATA_ROOT, f"{name}.tar.gz")
        for mirror in MIRRORS:
            if sh(f'curl -L --fail --retry 3 -C - -o "{tar}" {mirror}/{name}.tar.gz', check=False) == 0:
                break
        else:
            raise RuntimeError(f"{name}: OpenSLR 미러 세 곳 모두에서 받지 못했습니다.")
        digest = hashlib.md5()
        with open(tar, "rb") as fh:
            for block in iter(lambda: fh.read(1 << 22), b""):
                digest.update(block)
        if digest.hexdigest() != LIBRISPEECH_MD5[name]:
            tar.unlink()
            raise RuntimeError(f"{name}: md5가 OpenSLR 공개값과 다릅니다. 받은 파일을 지웠으니 이 셀을 다시 실행하세요.")
    staging = Path(DATA_ROOT, f".extract-{name}")
    shutil.rmtree(staging, ignore_errors=True)
    staging.mkdir(parents=True)
    sh(f'tar -xzf "{tar}" -C "{staging}"')
    Path(DATA_ROOT, "LibriSpeech").mkdir(exist_ok=True)
    for item in Path(staging, "LibriSpeech").iterdir():
        target = Path(DATA_ROOT, "LibriSpeech", item.name)
        if not target.exists():
            item.rename(target)
    shutil.rmtree(staging, ignore_errors=True)
    if tar != cached:
        if shutil.disk_usage(DRIVE_ROOT).free > tar.stat().st_size + 2e9:
            shutil.copy2(tar, cached.with_name(cached.name + ".part"))
            os.replace(cached.with_name(cached.name + ".part"), cached)
        else:
            print(f"Drive 여유 공간이 부족해 {name} 압축 파일을 캐시하지 않습니다 (다음 세션에 다시 받습니다).")
        tar.unlink()

# 필요한 방 음향 뱅크 확인 및 생성
need_pra = False
for c in CONFIG_LIST:
    _, _, args, _, _ = build_run_info(c)
    if sh(f"{sys.executable} train.py {args} --check-banks", check=False) == 3:
        need_pra = True
if need_pra:
    sh('pip -q install -c /content/constraints.txt '
       '"pyroomacoustics @ git+https://github.com/LCAV/pyroomacoustics@v0.8.5"')

for c in CONFIG_LIST:
    _, _, args, _, _ = build_run_info(c)
    sh(f"{sys.executable} train.py {args} --prepare-only")


## 4. 순차 학습 · 검증 실행

`CONFIG_LIST`에 있는 시스템들을 순서대로 실행합니다. 비교군은 임계값(단순 결합은 가중치도)만 검증 세트에서 정하고, 제안 모델은 학습한 뒤 최고 체크포인트(`best.pt`)와 그 임계값을 정합니다. 결과는 `DRIVE_ROOT/runs/<RUN_NAME>/`에 남고, 본 평가는 `benchmark.ipynb`가 이 폴더를 읽어 채점합니다.
- **이어하기(Resume) 자동 지원**: 이미 `summary.json`까지 완료된 시스템(학습 목표 스텝 도달)은 다시 학습하지 않고 즉시 넘어가며, 학습 도중 끊겼던 시스템은 `ckpt/LATEST.json`의 마지막 체크포인트에서 이어서 학습합니다.
- **새 커밋을 올린 뒤**: 끝나지 않은 제안 모델 학습은 체크포인트의 커밋과 코드가 다르면 이어 가지 않고 이 셀이 오류로 멈춥니다. 새 코드로 이어 가려면 `ALLOW_CODE_CHANGE = True`로 두고 다시 실행합니다(쓴 커밋은 모두 기록됩니다). 목표 걸음 수까지 끝난 실행은 막히지 않으며, 검증 기준이 바뀌었으면 더 학습하지 않고 새 코드로 다시 검증합니다.
- **`OVERRIDES`로 `train.max_steps`를 바꾸면**: 제안 모델의 실행 이름에 `-steps<N>`이 붙습니다(예: `proposed_carry-steps2000`). 시간 측정용 짧은 실행이 본 학습 폴더(`runs/proposed_carry`)를 이어 쓰지 않습니다. 무학습 비교군의 이름은 그대로입니다.


In [ ]:
from training.engine import run_is_complete

for c in CONFIG_LIST:
    run_name, run_dir, args, dev_only, cfg = build_run_info(c)
    print("\n" + "=" * 78)
    print(f">>> [{run_name}] 학습 및 검증 시작")
    print("=" * 78, flush=True)

    summary_path = Path(run_dir, "summary.json")
    already_trained = run_is_complete(run_dir, cfg)
    if already_trained:
        s_prev = json.loads(summary_path.read_text(encoding="utf-8"))
        print(f"[skip] {run_name}: 최종 설정(max_steps={cfg.train.max_steps})과 일치하는 완료된 summary.json이 Drive에 있습니다 "
              f"(steps={s_prev.get('trained_steps')}, best_step={s_prev.get('best_step')}).")
    else:
        sh(f"{sys.executable} train.py {args}")

# 마지막 실행된 CONFIG를 5절 상세 확인용 기본값으로 둔다
RUN_NAME, RUN_DIR, ARGS, DEV_CLEAN_ONLY, _ = build_run_info(CONFIG_LIST[-1])

## 5. 개별 실행 상세 확인

마지막에 실행된 시스템(또는 `RUN_DIR`로 지정한 시스템)의 검증 상세 지표와 학습 곡선을 확인합니다.

In [ ]:
summary = json.loads(Path(RUN_DIR, "summary.json").read_text(encoding="utf-8"))
official = (summary.get("official_n2") or {}).get("speaker_attribution_accuracy")
print(f"{summary['run']}  ({summary['system']}, {summary['trained_steps']} steps, best at {summary.get('best_step')})")
print(f"Spk Acc(cap4) {summary['spk_acc']:.4f} | 2인 공식(cap2, evaluate.py) {official} | causal 상한 {summary['causal_ceiling_by_N']}")
print()
gate = summary.get("gate_by_M", {})
cap2_by_m = summary.get("spk_acc_n2_cap2_by_M", {})
theta_off = summary.get("theta_official_by_M", summary["theta_by_M"])
print(f"{'마이크 수':>8}{'2인 공식(cap2)':>14}{'내부(cap4)':>11}{'교대 직후':>10}{'턴 안':>8}{'공식 임계값':>10}" + (f"{'gate':>8}" if gate else ""))
for M in sorted(summary["spk_acc_by_M"]):
    row = (f"{M:>8}{cap2_by_m.get(M, summary['spk_acc_by_M'][M]):>14.4f}{summary['spk_acc_by_M'][M]:>11.4f}"
           f"{summary.get('spk_acc_after_change_by_M', {}).get(M, float('nan')):>10.4f}"
           f"{summary.get('spk_acc_steady_by_M', {}).get(M, float('nan')):>8.4f}{theta_off[M]:>10.2f}")
    print(row + (f"{gate.get(M, float('nan')):>8.3f}" if gate else ""))
diag = summary.get("diagnostics")
if diag:
    print()
    print("진단 (같은 검증 세션, 마이크 수별 Spk Acc)")
    for name, key in (("목소리 전용", "voice_only_by_M"), ("목소리 전용 대비 이득", "paired_gain_over_voice_by_M"),
                      ("학습된 공간 경로만", "learned_space_only_by_M"), ("고정 게이트", "fixed_gate_by_M")):
        print(f"  {name:<14}", diag[key])
    print("  정지·무잡음 조건 이득  ", diag["static"]["paired_gain_over_voice_by_M"])
    print("  한 위치 조건 이득      ", diag["colocated"]["paired_gain_over_voice_by_M"])
    print("  한 위치 조건, 공간 경로만의 화자 분리", diag["colocated"]["spatial_part_talker_acc_by_M"],
          f"(우연 수준 약 {diag['colocated']['chance_approx']})")

metrics = [json.loads(line) for line in open(Path(RUN_DIR, "metrics.jsonl"), encoding="utf-8")]
train_rows = [m for m in metrics if m.get("kind") == "train"]
val_rows = [m for m in metrics if m.get("kind") == "val"]
if train_rows:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    for key in ("loss_spk", "loss_pos"):
        ax[0].plot([m["step"] for m in train_rows if key in m], [m[key] for m in train_rows if key in m], label=key)
    ax[0].set_title("train loss")
    ax[0].set_xlabel("step")
    ax[0].legend(fontsize=8)
    for M in "1234":
        pts = [(m["step"], m[f"gate_M{M}"]) for m in train_rows if f"gate_M{M}" in m]
        if pts:
            ax[1].plot(*zip(*pts), label=f"gate M={M}")
    if val_rows:
        ax[1].plot([m["step"] for m in val_rows], [m.get("spk_acc_n2_cap2", m["spk_acc"]) for m in val_rows], "k.-", label="val Spk Acc (cap2)")
    ax[1].set_xlabel("step")
    ax[1].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## 6. 전체 시스템 검증표

Drive에 저장된 모든 시스템의 검증셋 성적을 한눈에 비교합니다. 본 평가셋(`eval-t2`, `eval-t2-static`) 성적은 `benchmark.ipynb`에서 봅니다.


In [ ]:
import dataclasses
from training.config import DataConfig

# 학습 세션만 정하는 data 필드. 나머지 필드(검증 코퍼스·방·세션)가 표준값과 다르면 검증 세트가 다르다.
TRAIN_ONLY_DATA = {"train_root", "n_rooms_train", "n_speakers", "eval_layout_prob", "colocate_prob", "noise_snr_db",
                   "m_weights", "num_workers"}
STANDARD_DATA = json.loads(json.dumps(dataclasses.asdict(DataConfig())))


def other_val_set(run_dir: Path) -> bool:
    """실행이 저장한 config.json으로 표준 검증 세트와 다른 세트로 검증했는지 판단한다."""
    try:
        data = json.loads(Path(run_dir, "config.json").read_text(encoding="utf-8"))["config"]["data"]
    except (OSError, ValueError, KeyError, TypeError):
        # 설정 기록이 없는 실행만 이름으로 판단한다. RUN_TAG와 -steps<N>은 -devclean 뒤에 붙는다.
        return run_dir.name.startswith("smoke") or "-devclean" in run_dir.name
    train_root, val_root = (os.path.normpath(str(data.get(k, STANDARD_DATA[k]))) for k in ("train_root", "val_root"))
    if train_root == val_root:
        return True  # 한 코퍼스의 화자를 학습 70% · 검증 30%로 나눈다 (dev-clean 전용, smoke_real; training/corpus.py)
    return any(data.get(k, v) != v for k, v in STANDARD_DATA.items() if k not in TRAIN_ONLY_DATA)


print("=== [검증셋 (dev-clean, 2인 공식 cap=2)] ===")
rows = []
for path in sorted(Path(DRIVE_ROOT, "runs").glob("*/summary.json")):
    s = json.loads(path.read_text(encoding="utf-8"))
    rows.append((path.parent.name, s, path.parent))
print(f"{'run':<26}" + "".join(f"{'M=' + M:>9}" for M in "1234") + f"{'전체':>9}{'2인 공식':>10}")
for name, s, run_dir in rows:
    by_m = s.get("spk_acc_n2_cap2_by_M") or s.get("spk_acc_by_M", {})
    official = (s.get("official_n2") or {}).get("speaker_attribution_accuracy")
    other_set = other_val_set(run_dir)
    print(f"{name + (' *' if other_set else ''):<26}" + "".join(f"{by_m.get(M, float('nan')):>9.4f}" for M in "1234")
          + f"{s.get('spk_acc_n2_cap2', s.get('spk_acc', float('nan'))):>9.4f}{(official if official is not None else float('nan')):>10.4f}")
print("* 표준 검증 세트와 다른 세트로 검증한 실행 (실행의 config.json 기준: smoke 계열, dev-clean 전용 등)\n")


## 7. 검증 요약 블록 — 이 출력을 그대로 복사해 붙여 넣어 주세요

지금까지 Drive에 완료된 **모든 시스템의 검증셋 요약(`runs/`)**을 한 번에 출력합니다. 본 평가 결과 블록은 `benchmark.ipynb` 마지막 셀이 출력합니다.


In [ ]:
from train import SUMMARY_KEYS

all_runs = {}
for path in sorted(Path(DRIVE_ROOT, "runs").glob("*/summary.json")):
    s = json.loads(path.read_text(encoding="utf-8"))
    all_runs[path.parent.name] = {k: s.get(k) for k in SUMMARY_KEYS if s.get(k) is not None}

print("===== VC-SASR VALIDATION RESULTS (paste this back) =====")
print(json.dumps({"validation_runs": all_runs}, ensure_ascii=False))
print("===== END =====")
require_backup()
if storage is not None:
    storage.flush()
    print("개인 Drive 최종 백업 완료", flush=True)
